Import Libraries

In [18]:
import numpy as np
import pandas as pd
# from matplotlib import pyplot as plt
# import seaborn as sns
import plotly.express as px
#import plotly.graph_objects as go

Load Dataset

In [19]:
df = pd.read_csv("data.csv", encoding="latin1")

print(df.head())
print(df.shape)

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  
(541909, 8)


Dataset Overview

In [20]:
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicates:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None

Missing Values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

Duplicates: 5268


Missing Values Chart

In [21]:
missing = df.isnull().sum().reset_index()
missing.columns = ["Column", "Missing"]

missing = missing[missing["Missing"] > 0]

if len(missing) > 0:
    fig = px.bar(
        missing,
        x="Column",
        y="Missing",
        title="Missing Values"
    )
    fig.show()
else:
    print("No missing values found in dataset")

Data Cleaning

In [22]:
# Remove missing customer IDs
df = df.dropna(subset=["CustomerID"])

# Remove duplicates
df = df.drop_duplicates()

# Convert date
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Create Revenue column
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

print(df.shape)

(401604, 9)


Top 10 Products

In [29]:
top_products = (
    df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_products,
    x="Quantity",
    y="Description",
    orientation="h",
    title="Top 10 Products Sold"
)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(size=12, color="black"),
)

fig.show()

Top Countries by Revenue

In [30]:
country_sales = (
    df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    country_sales,
    x="Country",
    y="Revenue",
    title="Top Countries by Revenue"
)

fig.show()

Monthly Sales Trend

In [31]:
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = (
    df.groupby("Month")["Revenue"]
    .sum()
    .reset_index()
)

fig = px.line(
    monthly_sales,
    x="Month",
    y="Revenue",
    title="Monthly Revenue Trend",
    markers=True
)

fig.show()

Customer Spending Distribution

In [32]:
customer_sales = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .reset_index()
)

fig = px.histogram(
    customer_sales,
    x="Revenue",
    nbins=50,
    title="Customer Spending Distribution"
)

fig.show()

Top 10 Customers

In [33]:
top_customers = (
    df.groupby("CustomerID")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig = px.bar(
    top_customers,
    x="CustomerID",
    y="Revenue",
    title="Top 10 Customers"
)

fig.show()

Orders by Hour

In [34]:
df["Hour"] = df["InvoiceDate"].dt.hour

hourly_orders = (
    df.groupby("Hour")
    .size()
    .reset_index(name="Orders")
)

fig = px.line(
    hourly_orders,
    x="Hour",
    y="Orders",
    title="Orders by Hour"
)

fig.show()

Correlation Heatmap

In [35]:
numeric_df = df[["Quantity", "UnitPrice", "Revenue"]]

corr = numeric_df.corr()

fig = px.imshow(
    corr,
    text_auto=True,
    title="Correlation Heatmap"
)

fig.show()

KPI Summary

In [36]:
print("Total Revenue :", round(df["Revenue"].sum(), 2))
print("Total Orders :", df["InvoiceNo"].nunique())
print("Total Customers :", df["CustomerID"].nunique())
print("Total Products :", df["Description"].nunique())

Total Revenue : 8278519.42
Total Orders : 22190
Total Customers : 4372
Total Products : 3896
